<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 지식 증류 · 모델 경량화 · 05. Lottery Ticket Hypothesis

## The Lottery Ticket Hypothesis: Finding Sparse, Trainable Neural Networks

- **원 논문:** [The Lottery Ticket Hypothesis: Finding Sparse, Trainable Neural Networks](https://openreview.net/forum?id=rJl-b3RcF7)
- **저자 / 발표:** Jonathan Frankle, Michael Carbin · ICLR 2019 (2019)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** PyTorch tensor와 autograd, softmax·cross entropy, 선형대수 기초
- **로컬 학습 데이터:** `data/field_curriculum/compression_bench.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** dense network 학습으로 mask를 찾고 surviving weight를 초기값으로 되감아 sparse subnetwork를 재학습한다.

**축소하거나 생략한 부분:** 원 논문은 MNIST/CIFAR의 반복 magnitude pruning과 여러 seed를 비교했다. 실습은 한 번의 pruning과 작은 MLP다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2: winning ticket definition | Task 1 | 초기값·mask 보존 |
| §2, Figure 1: iterative pruning procedure | Task 1–2 | train→prune→rewind |
| §3: experimental methodology | Task 2 | 동일 mask·step과 세 fresh optimizer |
| §4: winning tickets results | Task 3 | rewind ticket/random sparse/dense 비교 |

## 미니 재현
중요한 것은 작은 weight 자체뿐 아니라, 그 연결에 대응하는 **초기값으로 되감는 것**이라는 가설을 코드로 분리합니다.

## 포트폴리오 해설: 정확도와 효율의 계약

$$f(\mathbf x,m\odot\theta_0)\quad\text{with}\quad\operatorname{Train}(m\odot\theta_0)
\approx\operatorname{Train}(\theta)$$

- **기호 정의:** m은 pruning mask, θ0는 원래 초기값, m⊙θ0는 winning ticket 후보입니다.
- **수식의 역할:** 학습 후 찾은 sparse 연결을 원래 초기값으로 되감아 독립적으로 재학습합니다.
- **구현 이유:** 정확도만 남기면 경량화의 비용 절감 주장을 검증할 수 없으므로,
  핵심 변환과 품질·비용 측정을 독립 함수와 method로 분리했습니다.
- **Task/코드 대응:** Task P의 논문 전용 class와 `forward`, `compute_loss`,
  `training_step`, `fit_portfolio_model`, `evaluate_portfolio_model`이
  §2: winning ticket definition의 흐름을 명시합니다.
- **입출력 shape:** 특징 [B, 32] → masked hidden [B, 64] → class logit [B, 4]
- **평가:** 동일 예산의 rewind ticket/random sparse/dense 정확도와 loss
- **원문 대비 한계:** 원 논문은 MNIST/CIFAR의 반복 magnitude pruning과 여러 seed를 비교했다. 실습은 한 번의 pruning과 작은 MLP다.

경량화 실험에서는 정확도만 보지 않고 parameter·sparsity·bit·latency 중 논문이
주장한 비용을 함께 기록해야 합니다. 아래 코드는 단계별 중간값을 남깁니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** features [B,D] + mask [O,I] → logits [B,K]와 sparsity scalar
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import math
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
torch.set_default_device(DEVICE)
print(ACCELERATOR.summary())


def locate(relative):
    for base in (Path.cwd(), *Path.cwd().parents):
        path = base / relative
        if path.exists():
            return path
    raise FileNotFoundError(relative)


dataset_path = locate("data/field_curriculum/compression_bench.npz")
raw = np.load(dataset_path, allow_pickle=False)
features = torch.from_numpy(raw["features"]).float().to(DEVICE)
labels = torch.from_numpy(raw["labels"]).long().to(DEVICE)
teacher_logits = torch.from_numpy(raw["teacher_logits"]).float().to(DEVICE)
teacher_weights = torch.from_numpy(raw["teacher_weights"]).float().to(DEVICE)
calibration = torch.from_numpy(raw["calibration_activations"]).float().to(DEVICE)
dense_weights = torch.from_numpy(raw["dense_weights"]).float().to(DEVICE)
train_idx = torch.from_numpy(raw["train_idx"]).long().to(DEVICE)
test_idx = torch.from_numpy(raw["test_idx"]).long().to(DEVICE)
images = torch.from_numpy(raw["images"]).float().to(DEVICE)
image_labels = torch.from_numpy(raw["image_labels"]).long().to(DEVICE)
image_train_idx = torch.from_numpy(raw["image_train_idx"]).long().to(DEVICE)
image_test_idx = torch.from_numpy(raw["image_test_idx"]).long().to(DEVICE)
token_embeddings = torch.from_numpy(raw["token_embeddings"]).float().to(DEVICE)
teacher_hidden = torch.from_numpy(raw["teacher_hidden"]).float().to(DEVICE)
teacher_attention = torch.from_numpy(raw["teacher_attention"]).float().to(DEVICE)
candidate_configs = torch.from_numpy(raw["candidate_configs"]).float()
assert features.shape == (512, 32) and teacher_logits.shape == (512, 4)
assert images.shape == (320, 1, 16, 16)
assert teacher_hidden.shape == (256, 6, 24)
print(
    dataset_path.name, "features", tuple(features.shape), "images", tuple(images.shape)
)

### 구현 단계 2 · 목적함수·학습 update

- **원문 위치:** §2: winning ticket definition / §2, Figure 1: iterative pruning procedure
- **이 셀의 책임:** Task 1 / Task 1–2
- **Tensor shape 계약:** features [B,D] + mask [O,I] → logits [B,K]와 sparsity scalar
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 초기값·mask 보존 / train→prune→rewind. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class TicketNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(32, 64)
        self.out = nn.Linear(64, 4)

    def forward(self, x):
        return self.out(F.relu(self.hidden(x)))


dense = TicketNet()
initial = {k: v.detach().clone() for k, v in dense.state_dict().items()}
optimizer = torch.optim.Adam(dense.parameters(), lr=0.025)
for _ in range(35):
    optimizer.zero_grad()
    loss = F.cross_entropy(dense(features[train_idx]), labels[train_idx])
    loss.backward()
    optimizer.step()
threshold = torch.quantile(dense.hidden.weight.detach().abs(), 0.70)
mask = (dense.hidden.weight.detach().abs() > threshold).float()
assert 0.28 < mask.mean().item() < 0.32

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §2: winning ticket definition / §2, Figure 1: iterative pruning procedure / §3: experimental methodology
- **이 셀의 책임:** Task 1 / Task 1–2 / Task 2
- **Tensor shape 계약:** features [B,D] + mask [O,I] → logits [B,K]와 sparsity scalar
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 초기값·mask 보존 / train→prune→rewind / 동일 mask·step과 세 fresh optimizer. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
ticket = TicketNet()
ticket.load_state_dict(initial)
with torch.no_grad():
    ticket.hidden.weight.mul_(mask)


def train_masked(model, mask, steps=45):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.025)
    history = []
    for _ in range(steps):
        optimizer.zero_grad()
        loss = F.cross_entropy(model(features[train_idx]), labels[train_idx])
        loss.backward()
        model.hidden.weight.grad.mul_(mask)
        optimizer.step()
        with torch.no_grad():
            model.hidden.weight.mul_(mask)
        history.append(float(loss.detach()))
    return history


ticket_history = train_masked(ticket, mask)
assert (ticket.hidden.weight.detach()[mask == 0] == 0).all() and ticket_history[
    -1
] < ticket_history[0]

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §4: winning tickets results
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** features [B,D] + mask [O,I] → logits [B,K]와 sparsity scalar
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** rewind ticket/random sparse/dense 비교. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
with torch.no_grad():
    ticket_accuracy = (
        (ticket(features[test_idx]).argmax(-1) == labels[test_idx])
        .float()
        .mean()
        .item()
    )
dense_count = sum(p.numel() for p in ticket.parameters())
active_count = (
    int(mask.sum())
    + ticket.hidden.bias.numel()
    + ticket.out.weight.numel()
    + ticket.out.bias.numel()
)
active_ratio = active_count / dense_count
assert ticket_accuracy > 0.75 and active_ratio < 0.45
plt.plot(ticket_history)
plt.title(
    f"rewound sparse ticket: acc={ticket_accuracy:.1%}, active={active_ratio:.1%}"
)
plt.show()
print(f"active parameters={active_ratio:.1%}, accuracy={ticket_accuracy:.1%}")

### Task P1 · 압축 대상과 핵심 모델

tensor shape와 비용 단위를 먼저 확인한 뒤 논문 전용 class의 `forward`와
`compute_loss`에 정확도·표현 보존 계약을 명시합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §2: winning ticket definition / §2, Figure 1: iterative pruning procedure / §3: experimental methodology / §4: winning tickets results
- **이 셀의 책임:** Task 1 / Task 1–2 / Task 2 / Task 3
- **Tensor shape 계약:** features [B,D] + mask [O,I] → logits [B,K]와 sparsity scalar
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 초기값·mask 보존 / train→prune→rewind / 동일 mask·step과 세 fresh optimizer / rewind ticket/random sparse/dense 비교. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    keep_ratio: float = 0.3
    learning_rate: float = 0.025
    mask_finding_steps: int = 8
    comparison_steps: int = 10


def prepare_batch() -> tuple[Tensor, Tensor]:
    return features[train_idx], labels[train_idx]


class LotteryTicketPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.linear = nn.Linear(features.shape[1], 48)
        self.classifier = nn.Linear(48, teacher_logits.shape[1])
        self._initial_state = {
            name: parameter.detach().clone()
            for name, parameter in self.named_parameters()
        }
        self._masks = {
            name: torch.ones_like(parameter)
            for name, parameter in self.named_parameters()
        }
        self.rewind_verified = False
        self.optimizer_reset_count = 0

    def forward(self, inputs: Tensor) -> Tensor:
        return self.classifier(F.relu(self.linear(inputs)))

    def compute_loss(self, logits: Tensor, targets: Tensor) -> Tensor:
        return F.cross_entropy(logits, targets)

    def training_step(self, inputs: Tensor, targets: Tensor) -> Tensor:
        return self.compute_loss(self(inputs), targets)

    def parameter_state(self) -> dict[str, Tensor]:
        return {
            name: parameter.detach().clone()
            for name, parameter in self.named_parameters()
        }

    def load_parameter_state(self, state: dict[str, Tensor]) -> None:
        with torch.no_grad():
            for name, parameter in self.named_parameters():
                parameter.copy_(state[name].to(parameter.device))

    def mask_state(self) -> dict[str, Tensor]:
        return {name: mask.detach().clone() for name, mask in self._masks.items()}

    def install_masks(self, masks: dict[str, Tensor]) -> None:
        self._masks = {
            name: masks[name].to(parameter.device).clone()
            for name, parameter in self.named_parameters()
        }
        self.enforce_mask()

    def prune_and_rewind(self) -> None:
        with torch.no_grad():
            prunable = [
                parameter.abs().flatten()
                for parameter in self.parameters()
                if parameter.ndim >= 2
            ]
            threshold = torch.quantile(
                torch.cat(prunable), 1.0 - self.config.keep_ratio
            )
            for name, parameter in self.named_parameters():
                if parameter.ndim >= 2:
                    self._masks[name] = (parameter.abs() >= threshold).float()
                initial = self._initial_state[name].to(parameter.device)
                mask = self._masks[name].to(parameter.device)
                parameter.copy_(initial * mask)
            self.rewind_verified = all(
                torch.equal(
                    parameter,
                    self._initial_state[name].to(parameter.device)
                    * self._masks[name].to(parameter.device),
                )
                for name, parameter in self.named_parameters()
            )

    def mask_gradients(self) -> None:
        for name, parameter in self.named_parameters():
            if parameter.grad is not None:
                parameter.grad.mul_(self._masks[name].to(parameter.device))

    def enforce_mask(self) -> None:
        with torch.no_grad():
            for name, parameter in self.named_parameters():
                parameter.mul_(self._masks[name].to(parameter.device))

    def sparsity(self) -> Tensor:
        weight_masks = [
            mask.flatten()
            for name, mask in self._masks.items()
            if name.endswith("weight")
        ]
        return 1.0 - torch.cat(weight_masks).mean()


@dataclass
class LotteryComparison:
    ticket: LotteryTicketPortfolioModel
    random_control: LotteryTicketPortfolioModel
    dense_baseline: LotteryTicketPortfolioModel
    mask_finding_history: list[float]
    histories: dict[str, list[float]]
    fresh_optimizer_count: int
    random_control_differs: bool

### Task P2 · 학습·pruning·distillation update

`fit_portfolio_model`에서 optimizer, mask/EMA/quantization 경계와 update 순서를
드러냅니다. 비용 절감이 학습 경로에 미치는 영향을 함께 읽으세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §2: winning ticket definition / §2, Figure 1: iterative pruning procedure / §3: experimental methodology
- **이 셀의 책임:** Task 1 / Task 1–2 / Task 2
- **Tensor shape 계약:** features [B,D] + mask [O,I] → logits [B,K]와 sparsity scalar
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 초기값·mask 보존 / train→prune→rewind / 동일 mask·step과 세 fresh optimizer. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def train_comparison_model(
    model: LotteryTicketPortfolioModel,
    inputs: Tensor,
    targets: Tensor,
    steps: int,
) -> tuple[list[float], int]:
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=model.config.learning_rate,
    )
    fresh_optimizer = int(len(optimizer.state) == 0)
    history = []
    for _ in range(steps):
        loss = model.training_step(inputs, targets)
        optimizer.zero_grad()
        loss.backward()
        model.mask_gradients()
        optimizer.step()
        model.enforce_mask()
        history.append(float(loss.detach().cpu()))
    model.optimizer_reset_count = fresh_optimizer
    return history, fresh_optimizer


def fit_portfolio_model(
    model: LotteryTicketPortfolioModel,
) -> LotteryComparison:
    inputs, targets = prepare_batch()
    initial_state = model.parameter_state()
    dense_optimizer = torch.optim.Adam(
        model.parameters(),
        lr=model.config.learning_rate,
    )
    mask_finding_history = []
    for _ in range(model.config.mask_finding_steps):
        loss = model.training_step(inputs, targets)
        dense_optimizer.zero_grad()
        loss.backward()
        dense_optimizer.step()
        mask_finding_history.append(float(loss.detach().cpu()))
    model.prune_and_rewind()

    shared_masks = model.mask_state()
    dense_baseline = LotteryTicketPortfolioModel(model.config)
    dense_baseline.load_parameter_state(initial_state)
    random_control = LotteryTicketPortfolioModel(model.config)
    random_control.install_masks(shared_masks)
    random_control_differs = any(
        not torch.equal(
            ticket_parameter,
            control_parameter,
        )
        for ticket_parameter, control_parameter in zip(
            model.parameters(),
            random_control.parameters(),
        )
    )

    histories = {}
    fresh_optimizer_count = 0
    candidates = {
        "ticket": model,
        "random_control": random_control,
        "dense_baseline": dense_baseline,
    }
    for name, candidate in candidates.items():
        history, fresh = train_comparison_model(
            candidate,
            inputs,
            targets,
            model.config.comparison_steps,
        )
        histories[name] = history
        fresh_optimizer_count += fresh
    return LotteryComparison(
        ticket=model,
        random_control=random_control,
        dense_baseline=dense_baseline,
        mask_finding_history=mask_finding_history,
        histories=histories,
        fresh_optimizer_count=fresh_optimizer_count,
        random_control_differs=random_control_differs,
    )

### Task P3 · 품질과 비용을 함께 평가

accuracy 또는 reconstruction 품질과 parameter·sparsity·bit·latency를 별도로
기록합니다. 로컬 지표는 원문 하드웨어 benchmark의 대체값이 아닙니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §4: winning tickets results
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** features [B,D] + mask [O,I] → logits [B,K]와 sparsity scalar
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** rewind ticket/random sparse/dense 비교. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    comparison: LotteryComparison,
) -> dict[str, float]:
    accuracies = {}
    with torch.no_grad():
        candidates = {
            "ticket": comparison.ticket,
            "random_control": comparison.random_control,
            "dense_baseline": comparison.dense_baseline,
        }
        for name, candidate in candidates.items():
            logits = candidate(features[test_idx])
            accuracy = (logits.argmax(1) == labels[test_idx]).float().mean()
            accuracies[name] = float(accuracy.cpu())
        ticket_sparsity = float(comparison.ticket.sparsity().cpu())
        control_sparsity = float(comparison.random_control.sparsity().cpu())
    ticket_loss = comparison.histories["ticket"][-1]
    control_loss = comparison.histories["random_control"][-1]
    dense_loss = comparison.histories["dense_baseline"][-1]
    winning_condition = (
        accuracies["ticket"] >= accuracies["random_control"]
        and accuracies["ticket"] >= accuracies["dense_baseline"] - 0.05
        and ticket_loss <= control_loss
    )
    return {
        "ticket_test_accuracy": accuracies["ticket"],
        "random_control_test_accuracy": accuracies["random_control"],
        "dense_baseline_test_accuracy": accuracies["dense_baseline"],
        "ticket_minus_random_accuracy": (
            accuracies["ticket"] - accuracies["random_control"]
        ),
        "ticket_minus_dense_accuracy": (
            accuracies["ticket"] - accuracies["dense_baseline"]
        ),
        "ticket_final_train_loss": ticket_loss,
        "random_control_final_train_loss": control_loss,
        "dense_baseline_final_train_loss": dense_loss,
        "random_minus_ticket_train_loss": control_loss - ticket_loss,
        "ticket_weight_sparsity": ticket_sparsity,
        "control_weight_sparsity": control_sparsity,
        "matched_comparison_steps": float(comparison.ticket.config.comparison_steps),
        "full_tensor_rewind_verified": float(comparison.ticket.rewind_verified),
        "fresh_comparison_optimizers": float(comparison.fresh_optimizer_count),
        "random_control_differs": float(comparison.random_control_differs),
        "winning_ticket_condition": float(winning_condition),
    }


portfolio_model = LotteryTicketPortfolioModel(PortfolioConfig())
portfolio_comparison = fit_portfolio_model(portfolio_model)
portfolio_metrics = evaluate_portfolio_model(portfolio_comparison)
all_histories = [
    portfolio_comparison.mask_finding_history,
    *portfolio_comparison.histories.values(),
]
assert all(np.isfinite(history).all() for history in all_histories)
assert portfolio_metrics["fresh_comparison_optimizers"] == 3.0
assert portfolio_metrics["random_control_differs"] == 1.0
assert (
    abs(
        portfolio_metrics["ticket_weight_sparsity"]
        - portfolio_metrics["control_weight_sparsity"]
    )
    < 1e-8
)
assert portfolio_metrics["winning_ticket_condition"] == 1.0
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 원 논문은 MNIST/CIFAR의 반복 magnitude pruning과 여러 seed를 비교했다. 실습은 한 번의 pruning과 작은 MLP다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.